Классификация текста

В данной работе решается задача многоклассовой классификации текста.  
Необходимо по тексту пользовательского отзыва определить его класс — оценку от 1 до 5.

Для решения задачи используется датасет `yelp_review_full` из библиотеки HuggingFace.


In [ ]:
import re
import string
import numpy as np
import pandas as pd

import nltk
from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, classification_report

import torch
from transformers import AutoTokenizer, AutoModel
from tqdm.notebook import tqdm

В рамках данной работы используются тексты на английском языке (датасет `yelp_review_full`).
Поэтому библиотеки для морфологического анализа русского языка, такие как `Natasha` и `pymorphy`,
устанавливаться и использоваться не будут.

Эти библиотеки предназначены для обработки русского текста (лемматизация, морфологический разбор,
выделение сущностей) и не применяются для английского корпуса.

Для предобработки текста в данной работе будет использоваться библиотека `NLTK`, которая содержит
инструменты для токенизации, удаления стоп-слов и лемматизации английского текста.
Это позволяет упростить код и не утяжелять вычисления лишними зависимостями.

In [ ]:
# загрузим модули nltk
#Удаление знаков препинания
nltk.download('punkt')

# Удаление стоп-слов
nltk.download('stopwords')

# Лексическая БД английского языка
nltk.download('wordnet')
nltk.download('omw-1.4')

[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package wordnet to /root/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package omw-1.4 to /root/nltk_data...
[nltk_data]   Package omw-1.4 is already up-to-date!


True

Загрузим датасет `yelp_review_full` из HuggingFace.  
Он содержит тексты отзывов пользователей и числовые метки классов, соответствующие оценкам от 1 до 5.

In [ ]:
nltk.download('punkt_tab')

[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!


True

In [ ]:
from datasets import load_dataset

ds = load_dataset("Yelp/yelp_review_full")
ds

DatasetDict({
    train: Dataset({
        features: ['label', 'text'],
        num_rows: 650000
    })
    test: Dataset({
        features: ['label', 'text'],
        num_rows: 50000
    })
})

Датасет уже содержит заранее подготовленное разделение на обучающую (train) и тестовую (test) выборки.
Это позволяет использовать обучающую часть для построения модели, а тестовую — для оценки качества.

In [ ]:
train_df = pd.DataFrame(ds["train"])
test_df = pd.DataFrame(ds["test"])

print("Размер train:", train_df.shape)
print("Размер test:", test_df.shape)
train_df.head(5)

Размер train: (650000, 2)
Размер test: (50000, 2)


,label,text
0,4,dr. goldberg offers everything i look for in a...
1,1,"Unfortunately, the frustration of being Dr. Go..."
2,3,Been going to Dr. Goldberg for over 10 years. ...
3,3,Got a letter in the mail last week that said D...
4,0,I don't know what Dr. Goldberg was like before...


In [ ]:
train_df.info()

train_df["label"].value_counts().sort_index()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 650000 entries, 0 to 649999
Data columns (total 2 columns):
 #   Column  Non-Null Count   Dtype 
---  ------  --------------   ----- 
 0   label   650000 non-null  int64 
 1   text    650000 non-null  object
dtypes: int64(1), object(1)
memory usage: 9.9+ MB


,count
label,
0,130000
1,130000
2,130000
3,130000
4,130000


	•	пропусков нет
	•	классы распределены равномерно
  
Метки принимают значения от 0 до 4, что соответствует оценкам от 1 до 5 звезд.

Распределение классов равномерное: в каждом классе по 130000 объектов.
Это означает, что датасет сбалансирован и не требует дополнительной балансировки.

In [ ]:
label_names = {
    0: "1 star",
    1: "2 stars",
    2: "3 stars",
    3: "4 stars",
    4: "5 stars"
}

train_df["label_name"] = train_df["label"].map(label_names)
test_df["label_name"] = test_df["label"].map(label_names)

train_df[["text", "label", "label_name"]].head()

,text,label,label_name
0,dr. goldberg offers everything i look for in a...,4,5 stars
1,"Unfortunately, the frustration of being Dr. Go...",1,2 stars
2,Been going to Dr. Goldberg for over 10 years. ...,3,4 stars
3,Got a letter in the mail last week that said D...,3,4 stars
4,I don't know what Dr. Goldberg was like before...,0,1 star


Датасет содержит:
- `text` — текст пользовательского отзыва,
- `label` — числовую метку класса.

Так как метки представлены числами от 0 до 4, дополнительно была выполнена расшифровка классов:
- 0 — 1 star
- 1 — 2 stars
- 2 — 3 stars
- 3 — 4 stars
- 4 — 5 stars

In [ ]:
#предобработка текста
stop_words = set(stopwords.words("english"))
lemmatizer = WordNetLemmatizer()

def preprocess_text(text):

    # нижний регистр
    text = text.lower()

    # удаляем цифры
    text = re.sub(r"\d+", " ", text)

    # удаляем пунктуацию
    text = re.sub(rf"[{re.escape(string.punctuation)}]", " ", text)

    # оставляем только буквы
    text = re.sub(r"[^a-zA-Z\s]", " ", text)

    # токенизация
    tokens = word_tokenize(text)

    # удаление стоп-слов, коротких слов и лемматизация
    tokens = [
        lemmatizer.lemmatize(token)
        for token in tokens
        if token not in stop_words and len(token) > 2
    ]

    # собираем обратно текст
    text = " ".join(tokens)

    # удаляем лишние пробелы
    text = re.sub(r"\s+", " ", text).strip()

    return text


In [ ]:
sample_texts = train_df["text"].head(5).apply(preprocess_text)
sample_texts

,text
0,goldberg offer everything look general practit...
1,unfortunately frustration goldberg patient rep...
2,going goldberg year think one patient started ...
3,got letter mail last week said goldberg moving...
4,know goldberg like moving arizona let tell sta...


Для предобработки текста были выполнены следующие шаги:
	•	приведение текста к нижнему регистру
	•	удаление цифр и знаков пунктуации
	•	токенизация текста
	•	удаление стоп-слов
	•	удаление коротких слов длиной менее 3 символов
	•	лемматизация слов
	•	удаление лишних пробелов


In [ ]:
#готовим рабочую подвыборку и очищаем её.
#650000 строк можно обрабатывать, но это долго. Для работы берем большую, но разумную выборку.

train_sample = train_df.sample(50000, random_state=42)
test_sample = test_df.sample(10000, random_state=42)

print("Размер train_sample:", train_sample.shape)
print("Размер test_sample:", test_sample.shape)
# Применяем предобработку
train_sample["clean_text"] = train_sample["text"].apply(preprocess_text)
test_sample["clean_text"] = test_sample["text"].apply(preprocess_text)

train_sample[["text", "clean_text", "label"]].head()

Размер train_sample: (50000, 3)
Размер test_sample: (10000, 3)


,text,clean_text,label
177288,"First of all i'm not a big fan of buffet, i tr...",first big fan buffet tried got credit staying ...,0
238756,Thanks Yelp. I was looking for the words to de...,thanks yelp looking word describe place meh se...,1
604225,Service was so-so. They were receiving a deliv...,service receiving delivery might food hot fres...,2
2838,Stamoolis Brothers is one of the Strip Distric...,stamoolis brother one strip district storefron...,2
586957,I want to give a 2 stars because the service s...,want give star service staff friendly good wel...,0


В результате очищенные тексты сохраняются в новый столбец clean_text, который далее используется для построения признаков и обучения моделей.


Преобразование текстов в числовые векторы с помощью TF-IDF

Для представления текстов в числовом виде используется `TfidfVectorizer` из библиотеки `scikit-learn`.

В рамках эксперимента рассматриваются разные параметры:
- `max_features` — максимальное количество признаков
- `ngram_range` — диапазон n-грамм

Для каждого варианта TF-IDF будет обучена модель логистической регрессии, после чего рассчитаны метрики качества:
`accuracy`, `precision`, `recall`, `f1-score`.

In [ ]:
y_train = train_sample["label"]#целевые переменные
y_test = test_sample["label"]

In [ ]:
tfidf_experiments = [ #список экспериментов
    {"name": "TF-IDF_1", "max_features": 10000, "ngram_range": (1, 1)},
    {"name": "TF-IDF_2", "max_features": 20000, "ngram_range": (1, 2)},
    {"name": "TF-IDF_3", "max_features": 30000, "ngram_range": (1, 2)},
]

In [ ]:
tfidf_results = []

for exp in tfidf_experiments:
    print("=" * 60)
    print(f'Эксперимент: {exp["name"]}')
    print(f'max_features = {exp["max_features"]}')
    print(f'ngram_range = {exp["ngram_range"]}')

    vectorizer = TfidfVectorizer(
        max_features=exp["max_features"],
        ngram_range=exp["ngram_range"]
    )

    X_train = vectorizer.fit_transform(train_sample["clean_text"])
    X_test = vectorizer.transform(test_sample["clean_text"])

    print("Размер X_train:", X_train.shape)
    print("Размер X_test:", X_test.shape)

    model = LogisticRegression(max_iter=1000)
    model.fit(X_train, y_train)

    y_pred = model.predict(X_test)

    accuracy = accuracy_score(y_test, y_pred)
    precision = precision_score(y_test, y_pred, average="weighted")
    recall = recall_score(y_test, y_pred, average="weighted")
    f1 = f1_score(y_test, y_pred, average="weighted")

    print("Accuracy:", accuracy)
    print("Precision:", precision)
    print("Recall:", recall)
    print("F1-score:", f1)

    tfidf_results.append({
        "Model": "LogisticRegression",
        "Vectorizer": exp["name"],
        "max_features": exp["max_features"],
        "ngram_range": exp["ngram_range"],
        "Accuracy": accuracy,
        "Precision": precision,
        "Recall": recall,
        "F1-score": f1
    })

Эксперимент: TF-IDF_1
max_features = 10000
ngram_range = (1, 1)
Размер X_train: (50000, 10000)
Размер X_test: (10000, 10000)
Accuracy: 0.5485
Precision: 0.5437147962492958
Recall: 0.5485
F1-score: 0.5454697821285559
Эксперимент: TF-IDF_2
max_features = 20000
ngram_range = (1, 2)
Размер X_train: (50000, 20000)
Размер X_test: (10000, 20000)
Accuracy: 0.5549
Precision: 0.5509029645915573
Recall: 0.5549
F1-score: 0.5524330087883982
Эксперимент: TF-IDF_3
max_features = 30000
ngram_range = (1, 2)
Размер X_train: (50000, 30000)
Размер X_test: (10000, 30000)
Accuracy: 0.5586
Precision: 0.554710710562904
Recall: 0.5586
F1-score: 0.5561691001471806


In [ ]:
#таблица результатов и лучшие метрики
tfidf_results_df = pd.DataFrame(tfidf_results)
tfidf_results_df

tfidf_results_df.sort_values(by="F1-score", ascending=False)

,Model,Vectorizer,max_features,ngram_range,Accuracy,Precision,Recall,F1-score
2,LogisticRegression,TF-IDF_3,30000,"(1, 2)",0.5586,0.554711,0.5586,0.556169
1,LogisticRegression,TF-IDF_2,20000,"(1, 2)",0.5549,0.550903,0.5549,0.552433
0,LogisticRegression,TF-IDF_1,10000,"(1, 1)",0.5485,0.543715,0.5485,0.545470


Эксперименты показали, что увеличение числа признаков и использование биграмм улучшает качество модели.
Лучший результат был получен при max_features = 30000 и ngram_range = (1,2).
При этом точность остаётся относительно невысокой, поскольку TF-IDF учитывает только частоты слов и не отражает семантический контекст текста.

In [ ]:
#Использование предобученной модели BERT и получение эмбеддингов предложений.

model_name = "bert-base-uncased"

tokenizer = AutoTokenizer.from_pretrained(model_name)
bert_model = AutoModel.from_pretrained(model_name)

bert_model.eval()

config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/440M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: bert-base-uncased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


BertModel(
  (embeddings): BertEmbeddings(
    (word_embeddings): Embedding(30522, 768, padding_idx=0)
    (position_embeddings): Embedding(512, 768)
    (token_type_embeddings): Embedding(2, 768)
    (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
    (dropout): Dropout(p=0.1, inplace=False)
  )
  (encoder): BertEncoder(
    (layer): ModuleList(
      (0-11): 12 x BertLayer(
        (attention): BertAttention(
          (self): BertSelfAttention(
            (query): Linear(in_features=768, out_features=768, bias=True)
            (key): Linear(in_features=768, out_features=768, bias=True)
            (value): Linear(in_features=768, out_features=768, bias=True)
            (dropout): Dropout(p=0.1, inplace=False)
          )
          (output): BertSelfOutput(
            (dense): Linear(in_features=768, out_features=768, bias=True)
            (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
            (dropout): Dropout(p=0.1, inplace=False)
  

In [ ]:
def get_bert_embeddings(texts, batch_size=64):

    embeddings = []

    for i in tqdm(range(0, len(texts), batch_size)):

        batch = texts[i:i+batch_size]

        encoded = tokenizer(
            batch,
            padding=True,
            truncation=True,
            max_length=64,
            return_tensors="pt"
        )

        with torch.no_grad():
            outputs = bert_model(**encoded)

        cls_embeddings = outputs.last_hidden_state[:,0,:]

        embeddings.append(cls_embeddings.numpy())

    return np.vstack(embeddings)

In [ ]:
bert_train = train_sample.sample(5000, random_state=42)
bert_test = test_sample.sample(1000, random_state=42)

Получение эмбеддингов с помощью BERT

Для второй части эксперимента используется предобученная модель `bert-base-uncased` из библиотеки `transformers`.

Поскольку получение эмбеддингов с помощью BERT требует значительно больше вычислительных ресурсов, чем TF-IDF, для ускорения работы была уменьшена выборка данных.  
Кроме того, были уменьшены:
- максимальная длина текста (`max_length`)
- размер батча (`batch_size`)

Это позволило сократить время вычислений, сохранив при этом корректность эксперимента и возможность сравнения результатов с классическим подходом.

In [ ]:
X_train_bert = get_bert_embeddings(list(bert_train["clean_text"]))
X_test_bert = get_bert_embeddings(list(bert_test["clean_text"]))

y_train_bert = bert_train["label"]
y_test_bert = bert_test["label"]

  0%|          | 0/79 [00:00<?, ?it/s]

  0%|          | 0/16 [00:00<?, ?it/s]

In [ ]:
bert_clf = LogisticRegression(max_iter=1000)

bert_clf.fit(X_train_bert, y_train_bert)

bert_pred = bert_clf.predict(X_test_bert)

In [ ]:
bert_accuracy = accuracy_score(y_test_bert, bert_pred)
bert_precision = precision_score(y_test_bert, bert_pred, average="weighted")
bert_recall = recall_score(y_test_bert, bert_pred, average="weighted")
bert_f1 = f1_score(y_test_bert, bert_pred, average="weighted")

print("BERT + LogisticRegression")
print("Accuracy:", bert_accuracy)
print("Precision:", bert_precision)
print("Recall:", bert_recall)
print("F1-score:", bert_f1)

BERT + LogisticRegression
Accuracy: 0.448
Precision: 0.45119803803752284
Recall: 0.448
F1-score: 0.4491194294561747


In [ ]:
print(classification_report(y_test_bert, bert_pred))

              precision    recall  f1-score   support

           0       0.60      0.60      0.60       206
           1       0.41      0.40      0.40       204
           2       0.38      0.34      0.36       207
           3       0.29      0.34      0.31       178
           4       0.56      0.55      0.55       205

    accuracy                           0.45      1000
   macro avg       0.45      0.45      0.45      1000
weighted avg       0.45      0.45      0.45      1000



In [ ]:
bert_result = pd.DataFrame([{
    "Model": "LogisticRegression",
    "Vectorizer": "BERT embeddings",
    "max_features": None,
    "ngram_range": None,
    "Accuracy": bert_accuracy,
    "Precision": bert_precision,
    "Recall": bert_recall,
    "F1-score": bert_f1
}])

all_results = pd.concat([tfidf_results_df, bert_result], ignore_index=True)
all_results

,Model,Vectorizer,max_features,ngram_range,Accuracy,Precision,Recall,F1-score
0,LogisticRegression,TF-IDF_1,10000,"(1, 1)",0.5485,0.543715,0.5485,0.545470
1,LogisticRegression,TF-IDF_2,20000,"(1, 2)",0.5549,0.550903,0.5549,0.552433
2,LogisticRegression,TF-IDF_3,30000,"(1, 2)",0.5586,0.554711,0.5586,0.556169
3,LogisticRegression,BERT embeddings,None,None,0.4480,0.451198,0.4480,0.449119


In [ ]:
all_results.sort_values(by="F1-score", ascending=False)

,Model,Vectorizer,max_features,ngram_range,Accuracy,Precision,Recall,F1-score
2,LogisticRegression,TF-IDF_3,30000,"(1, 2)",0.5586,0.554711,0.5586,0.556169
1,LogisticRegression,TF-IDF_2,20000,"(1, 2)",0.5549,0.550903,0.5549,0.552433
0,LogisticRegression,TF-IDF_1,10000,"(1, 1)",0.5485,0.543715,0.5485,0.545470
3,LogisticRegression,BERT embeddings,None,None,0.4480,0.451198,0.4480,0.449119


Вывод по результатам экспериментов

В качестве базового подхода использовалось преобразование текстов в числовые признаки с помощью TF-IDF, а также более современный подход с использованием эмбеддингов предобученной модели BERT.

Было проведено несколько экспериментов с TF-IDF, в которых изменялись параметры `max_features` и `ngram_range`. Результаты показали, что увеличение количества признаков и использование биграмм позволяет немного улучшить качество модели. Лучший результат среди TF-IDF моделей показал вариант TF-IDF_3 с параметрами `max_features = 30000` и `ngram_range = (1,2)`. Для данной модели значение F1-score составило 0.556.

Также была проведена классификация с использованием эмбеддингов модели BERT. Несмотря на то, что BERT является более современным подходом, в данном эксперименте его качество оказалось ниже F1-score ≈ 0.449. Это можно объяснить тем, что использовались готовые эмбеддинги без дообучения модели, а также была уменьшена обучающая выборка для ускорения вычислений.

Таким образом, в данной работе лучшей моделью оказалась модель на основе TF-IDF, которая показала наибольшие значения Accuracy и F1-score. При этом BERT-подход обладает потенциалом для улучшения качества при использовании более крупных выборок и полноценного fine-tuning модели.